In [9]:
!pip install faiss-cpu tiktoken pypdf python-dotenv openai gradio --quiet
!pip install sentence-transformers --quiet


In [2]:
import os
import glob
import faiss
import numpy as np
from pypdf import PdfReader
from openai import OpenAI
import gradio as gr
import tiktoken

In [ ]:
os.environ["OPENAI_API_KEY"] = "api-key here"
client = OpenAI()

In [16]:
def load_pdf(pdf_folder="/content"):
  documents = []
  filenames = []

  pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
  print("found pdfs:", pdf_files)

  for pdf in pdf_files:
    reader = PdfReader(pdf)
    text = ""
    for page in reader.pages:
      text += page.extract_text() + "\n"

    documents.append(text)
    filenames.append(os.path.basename(pdf))

  return documents, filenames

docs, filenames = load_pdf()
print("Loaded", len(docs), "PDFs")


found pdfs: ['/content/medical_pdf3.pdf', '/content/medical_pdf4.pdf', '/content/medical_pdf2.pdf', '/content/medical_pdf.pdf']
Loaded 4 PDFs


In [17]:
def chunk_text(text, chunk_size=500, overlap=100):
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]
    chunks.append(chunk)
    start += chunk_size - overlap
  return chunks

all_chunks = []
chunk_sources = []

for doc, file in zip(docs, filenames):
  chunks = chunk_text(doc)
  all_chunks.extend(chunks)
  chunk_sources.extend([file] * len(chunks))

print("Total chunks: ", len(all_chunks))

Total chunks:  6627


In [10]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")   # free & fast


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
def get_embeddings(text_list):
    return embed_model.encode(text_list, convert_to_numpy=True)


In [19]:
embeddings = get_embeddings(all_chunks)
embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [20]:
def retrieve(query, k=5):
  query_emb = np.array(get_embeddings([query])[0]).astype("float32")
  D, I = index.search(query_emb.reshape(1, -1), k)

  retrieved_chunks = [all_chunks[i] for i in I[0]]
  retrieved_sources = [chunk_sources[i] for i in I[0]]

  return retrieved_chunks, retrieved_sources

def generate_answer(query):
  chunks, sources = retrieve(query)

  context = ""
  for i, chunk in enumerate(chunks):
    context += f"source {i+1} ({sources[i]}):\n{chunk}\n\n"

  prompt = f""" you are a medical assistant. Answer the user's question strictly using

  CONTEXT:
  {context}

  QUESTION:
  {query}

  Provide a clear answer and mention sources at the end.
  """

  completion = client.chat.completions.create(
      model="gpt-4o-mini",
      messages=[{"role": "user", "content": prompt}],
  )

  return completion.choices[0].message.content

In [21]:
def chat_fn(message, history):
  answer = generate_answer(message)
  return answer


ui = gr.ChatInterface(
    fn = chat_fn, title="Hospital AI Assistant",
    description="Ask health and hospital related questions"
)

ui.launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://86ddb266df8a1f3cb8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
